# LAB 12 - Random Forest for Regression

In this lab we will be extending the previous lab about Decision trees and build a Regression model using Random Forest.

For simplicity, we will be using the same dataset as the previous lab (you can find it in ECLASS).

**IMPORTANT:** For this lab, if you haven't finished your code from last week's lab on Decision trees, you will have the option to use the sklearn implementation for a regression tree. However, this doesn't mean that you should skip the previous lab. This is just so that you don't get behind with the content and you don't spend all your time today working on the previous lab. 

In [15]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

As mentioned before, use the Boston Housing data and prepare your train/val/test split as usual.

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [17]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_csv("housing.txt", names=housing_names, sep='\\s+')
data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


The target variable will be as usual `MEDV`. Use the rest as features.

In [18]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values

In [19]:
data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48,22.0


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=0)

## Exercise 1 -- Bootstrap

Also known as [bagging](https://en.wikipedia.org/wiki/Bootstrap_aggregating), this technique consists of making several samples with replacement of the original data, using each of the samples to train an estimator, and then aggregating the predictions using the average (this is also a type of model ensemble).

In [21]:
def bootstrap(X, num_bags=10):
    """
    Given a dataset and a number of bags,
    sample the dataset with replacement.
    
    This function does not return a copy
    of the datapoints, but a list of indices
    with compatible dimensionality
    
    Parameters
    ----------
    X : ndarray
        A dataset
    num_bags : int, default 10
        The number of bags to create
    
    Returns
    -------
    list of ndarray
        The list contains `num_bags` integer one-dimensional ndarrays.
        Each of these contains the indices corresponding to the 
        sampled datapoints in `X`
    
    Notes
    -----
    * The number of datapoints in each bach will
      match the number of datapoints in the given
      dataset.
    * The
    """
    rng = np.random.default_rng(0) # you can change the seed, or use 0 to replicate my results
    N = X.shape[0]
    return [rng.choice(N, size=N) for _ in range(num_bags)]

In [22]:
rng = np.random.default_rng(0)
X_small = rng.random(size=(100,2))
bags = bootstrap(X_small)
bags[0]

array([85, 63, 51, 26, 30,  4,  7,  1, 17, 81, 64, 91, 50, 60, 97, 72, 63,
       54, 55, 93, 27, 81, 67,  0, 39, 85, 55,  3, 76, 72, 84, 17,  8, 86,
        2, 54,  8, 29, 48, 42, 40,  2,  0, 12,  0, 67, 52, 64, 25, 61, 76,
       38, 46, 99, 80, 98, 37, 68, 95, 65, 84, 68, 70, 38, 87, 13, 57, 72,
       84, 52, 37, 31, 42, 48, 71, 88,  7, 93, 53, 35, 67, 57, 25, 32, 71,
       59, 50, 33, 76, 39, 32, 89, 26, 22, 71, 62,  4,  8, 37, 83],
      dtype=int64)

## Exercise 2 -- Aggregation

The second part of bagging.

In [23]:
def aggregate_regression(preds):
    """
    Aggregate predictions by several estimators
    
    Parameters
    ----------
    preds : list of ndarray
        Predictions from multiple estimators.
        All ndarrays in this list should have the same
        dimensionality.
        
    Return
    ------
    ndarray
        The mean of the predictions
    """
    return np.vstack(preds).mean(axis=0) # axis 0 is crucial because we are aggregating the predictions across estimators, not all the predictions for each estimator. 

In [24]:
# dummy NDARRAY of predictions to test the function
preds = [np.array([1, 2, 3]), np.array([4, 5, 6]), np.array([7, 8, 9])]
# test the function
agg_preds = aggregate_regression(preds)
agg_preds


array([4., 5., 6.])

## Exercise 3 -- Random Forest for regression

Using the functions you implemented above, it is now time to put all of them together to train several decision trees and then ensemble them to output a single prediction. For the random forest, however, we need to select a subset of features at each split on the decision tree. 

For this part, you can use the sklearn implementation of Decision trees for regression as your estimator for each set of features and bags. See below an example of how to do this, and always remember to check the necessary documentation when using an external function.

Some parameters you will have to set are: 
* num_features: number of features per estimator
* min_samples: min number of samples per leaf node
* max_depth: maximum depth of the decision tree (each estimator)
* num_estimators: number of decision trees you will create using each bag and random set of features

In [25]:
# example of sklearn Decision tree
estimator = DecisionTreeRegressor(max_depth=10)
estimator.fit(X, y)
estimator.predict(X)

array([24.        , 21.4625    , 34.8       , 33.2       , 37.        ,
       28.7       , 20.14074074, 27.1       , 16.5       , 18.9       ,
       15.        , 18.16666667, 21.7       , 20.14074074, 19.97      ,
       20.14074074, 22.66666667, 17.5       , 21.70909091, 18.16666667,
       13.        , 18.16666667, 15.95      , 14.34      , 17.65294118,
       13.5       , 17.65294118, 14.61666667, 19.97      , 21.        ,
       12.86666667, 14.5       , 12.86666667, 14.61666667, 13.        ,
       21.70909091, 21.70909091, 21.70909091, 21.70909091, 30.8       ,
       34.8       , 26.6       , 24.30625   , 24.30625   , 20.14074074,
       18.14      , 20.14074074, 16.6       , 14.4       , 19.4       ,
       20.14074074, 21.4625    , 24.30625   , 20.14074074, 18.9       ,
       35.4       , 24.15      , 31.6       , 23.34      , 20.14074074,
       18.14      , 16.        , 22.55      , 24.66666667, 32.85      ,
       24.15      , 20.14074074, 20.14074074, 18.14      , 20.14

In [26]:
def root_mean_squared_error(y_true, y_pred):
    """
    Calculate the root mean squared error between true and predicted values.

    Parameters
    ----------
    y_true : ndarray
        True target values.
    y_pred : ndarray
        Predicted target values.

    Returns
    -------
    float
        The root mean squared error.
    """
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [27]:
class RandomForestRegressor:
    def __init__(self, n_estimators=10, max_depth=None, num_features=None, min_samples=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.num_features = num_features
        self.min_samples = min_samples
        self.trees = []

    def fit(self, X, y):
        bags = bootstrap(X, num_bags=self.n_estimators)
        for bag in bags:
            # Randomly select a subset of features
            rng = np.random.default_rng(0)
            feature_indices = rng.choice(X.shape[1], size=self.num_features, replace=False)
            X_subset = X[bag][:, feature_indices]
            tree = DecisionTreeRegressor(max_depth=self.max_depth, min_samples_leaf=self.min_samples)
            tree.fit(X_subset, y[bag])
            self.trees.append((tree, feature_indices))

    def predict(self, X):
        preds = []
        for tree, feature_indices in self.trees:
            X_subset = X[:, feature_indices]
            preds.append(tree.predict(X_subset))
        return aggregate_regression(preds)

# Use it on the training set and evaluate on the validation set.
rf = RandomForestRegressor(n_estimators=10, max_depth=5, num_features=3, min_samples=4)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

# Calculate RMSE on the validation set
rmse_val = root_mean_squared_error(y_val, y_pred)
print(f"RMSE on validation set: {rmse_val:.2f}")


RMSE on validation set: 7.12


In [28]:
# find the best hyperparameters manually. 
# You can use a grid search or random search for this, but here we will do it manually.
max_depths = [3, 5, 10]
num_features_list = [4, 8, X_train.shape[1]]  # 4, 8 or all features
best_rmse = float('inf')
min_samples = [2,3,5]
n_estimators = [5, 10, 20]
for max_depth in max_depths:
    for num_features in num_features_list:
        for min_sample in min_samples:
            for n_estimator in n_estimators:
                rf = RandomForestRegressor(n_estimators=n_estimator, max_depth=max_depth, num_features=num_features, min_samples=min_sample)
                rf.fit(X_train, y_train)
                y_pred = rf.predict(X_val)
                rmse_val = root_mean_squared_error(y_val, y_pred)
                print(f"RMSE on validation set with max_depth={max_depth}, num_features={num_features}, min_samples={min_sample}, n_estimators={n_estimator}: {rmse_val:.2f}")
                
                if rmse_val < best_rmse:
                    best_rmse = rmse_val
                    best_params = (max_depth, num_features, min_sample, n_estimator)

print(f"Best RMSE: {best_rmse:.2f} with parameters: max_depth={best_params[0]}, num_features={best_params[1]}, min_samples={best_params[2]}, n_estimators={best_params[3]}")
# Final model with best hyperparameters
rf_final = RandomForestRegressor(n_estimators=best_params[3], max_depth=best_params[0], num_features=best_params[1], min_samples=best_params[2])
# join the training and validation sets to train the final model
X_final = np.vstack((X_train, X_val))
y_final = np.hstack((y_train, y_val))
rf_final.fit(X_final, y_final)
# Evaluate on the test set
y_test_pred = rf_final.predict(X_test)
# Calculate RMSE on the test set
rmse_test = root_mean_squared_error(y_test, y_test_pred)
print(f"RMSE on test set: {rmse_test:.2f}")

RMSE on validation set with max_depth=3, num_features=4, min_samples=2, n_estimators=5: 7.10
RMSE on validation set with max_depth=3, num_features=4, min_samples=2, n_estimators=10: 7.45
RMSE on validation set with max_depth=3, num_features=4, min_samples=2, n_estimators=20: 7.28
RMSE on validation set with max_depth=3, num_features=4, min_samples=3, n_estimators=5: 7.21
RMSE on validation set with max_depth=3, num_features=4, min_samples=3, n_estimators=10: 7.50
RMSE on validation set with max_depth=3, num_features=4, min_samples=3, n_estimators=20: 7.35
RMSE on validation set with max_depth=3, num_features=4, min_samples=5, n_estimators=5: 7.63
RMSE on validation set with max_depth=3, num_features=4, min_samples=5, n_estimators=10: 7.70
RMSE on validation set with max_depth=3, num_features=4, min_samples=5, n_estimators=20: 7.53
RMSE on validation set with max_depth=3, num_features=8, min_samples=2, n_estimators=5: 4.84
RMSE on validation set with max_depth=3, num_features=8, min_sam